# 🎗️ 02 · Cómo trabaja un detector público

**Después del cuaderno 01 · 15–20 minutos**

Vamos a seguir una mamografía paso a paso: veremos **qué imagen recibe el modelo**, cómo la preparamos, cómo pedimos una predicción y cómo se dibuja el resultado. Después haremos lo mismo con **cuatro casos de masas** para ver resultados diferentes.

Usamos un modelo público YOLO11n de [Digital Eye for Mammography](https://github.com/cbddobvyz/digitaleye-mammography/releases/tag/shared-models.v2). Este modelo busca **masas**; por eso los cuatro casos de calcificaciones quedan fuera de esta actividad. El recuadro amarillo será una propuesta del modelo y el rojo será la referencia de CBIS-DDSM. Una propuesta no es un diagnóstico.

## Preparación

Ejecuta esta celda una vez. En Colab, sube `taller_colab.zip` si aparece la solicitud. El ZIP trae los PNG y el archivo del modelo. La celda comprueba que el modelo incluido sea el esperado y lo carga.

In [ ]:
from pathlib import Path
import hashlib
import io
import json
import subprocess
import sys
import zipfile

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
from PIL import Image

try:
    from ultralytics import YOLO
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics==8.4.161", "-q"])
    from ultralytics import YOLO

carpeta = next(
    (p for p in [Path("."), Path(".."), Path("/content")]
     if (p / "data/taller/casos.json").exists()
     and (p / "models/digitaleye_yolo11_n.pt").exists()),
    None,
)

if carpeta is None:
    from google.colab import files
    print("Sube taller_colab.zip")
    archivos = files.upload()
    nombre = next((n for n in archivos if n.endswith(".zip")), None)
    if nombre is None:
        raise ValueError("Selecciona taller_colab.zip para continuar.")
    with zipfile.ZipFile(io.BytesIO(archivos[nombre])) as paquete:
        paquete.extractall("/content")
    carpeta = Path("/content")

casos = json.loads((carpeta / "data/taller/casos.json").read_text(encoding="utf-8"))
ruta_modelo = carpeta / "models/digitaleye_yolo11_n.pt"
huella = hashlib.sha256(ruta_modelo.read_bytes()).hexdigest()
if huella != "4c6ed69c85775bfe7642a428489fc0a90f53cb3ac7b164eb7da7220ef9549a14":
    raise ValueError("El archivo del modelo no coincide con el del taller.")

modelo = YOLO(str(ruta_modelo))
print("Modelo cargado. Ya podemos usar los casos de masas.")

## Paso 1 · Abrir la imagen

Empezamos con `masa_1`. El PNG contiene los píxeles de la mamografía en tonos grises. `casos.json` nos dice cuál archivo corresponde al caso y dónde está el recuadro de referencia, pero **la referencia todavía no se entrega al modelo**.

In [ ]:
caso = casos[0]  # masa_1
ruta_imagen = carpeta / "data/taller" / caso["imagen"]
imagen = np.array(Image.open(ruta_imagen).convert("L"))

print("Caso:", caso["id"])
print("Tamaño de la imagen:", imagen.shape, "píxeles")
plt.figure(figsize=(6, 8))
plt.imshow(imagen, cmap="gray", vmin=0, vmax=255)
plt.axis("off")
plt.show()

## Paso 2 · Preparar lo que recibe el modelo

La imagen tiene **un canal** porque está en grises. El detector espera **tres canales**. Copiamos los mismos tonos de gris tres veces; no inventamos colores ni modificamos el PNG guardado. La forma de `entrada` indica alto, ancho y tres canales.

In [ ]:
entrada = np.repeat(imagen[:, :, None], 3, axis=2)
print("Imagen en grises:", imagen.shape)
print("Entrada del modelo:", entrada.shape)

## Paso 3 · Pedir una predicción

`modelo.predict` analiza la imagen. `imgsz=1024` es el tamaño de trabajo que usa el detector y `conf=0.05` permite mostrar también propuestas con puntuación baja. La salida trae recuadros y una puntuación para cada propuesta. **Esa puntuación no es la probabilidad de cáncer.**

In [ ]:
resultado = modelo.predict(
    entrada, imgsz=1024, conf=0.05, device="cpu", verbose=False
)[0]

print("Propuestas encontradas:", len(resultado.boxes))
for caja in resultado.boxes:
    print("Puntuación:", round(float(caja.conf[0]), 2),
          "· recuadro [izquierda, arriba, derecha, abajo]:",
          [round(x) for x in caja.xyxy[0].tolist()])

## Paso 4 · Ver el resultado

Dibujamos en **amarillo** la propuesta con mayor puntuación. El recuadro **rojo** se añade después: viene de las coordenadas de referencia guardadas en `casos.json`. Así podemos comparar ambos sin que la anotación haya entrado al modelo.

In [ ]:
def dibujar_resultado(caso: dict, imagen: np.ndarray, resultado: object, eje: object) -> None:
    """Dibuja la anotación de referencia y la mejor propuesta del modelo."""
    # Mostrar la imagen en escala de grises.
    eje.imshow(imagen, cmap="gray", vmin=0, vmax=255)

    # Recuadro real indicado en el JSON de referencia.
    izquierda, arriba, derecha, abajo = caso["recuadro_referencia"]
    eje.add_patch(
        Rectangle(
            (izquierda, arriba),
            derecha - izquierda,
            abajo - arriba,
            fill=False,
            edgecolor="red",
            linewidth=3,
        )
    )

    # Recuadro predicho por el modelo. Se elige la caja con mayor puntuación.
    cajas = resultado.boxes
    if len(cajas):
        mejor = int(cajas.conf.argmax())
        x1, y1, x2, y2 = cajas.xyxy[mejor].cpu().numpy()
        eje.add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                edgecolor="yellow",
                linewidth=3,
            )
        )

    eje.set_title(f"{caso['id']} · {len(cajas)} propuesta(s)")
    eje.axis("off")


# Generar la figura y mostrar el resultado final.
figura, eje = plt.subplots(figsize=(6, 8))
dibujar_resultado(caso, imagen, resultado, eje)
plt.show()

## Paso 5 · Repetir con más imágenes

Aplicamos los mismos tres pasos a las **cuatro mamografías de masas**: abrir el PNG, preparar tres canales y pedir una predicción. La cuadrícula se crea al ejecutar esta celda. Repetir con imágenes distintas permite ver resultados distintos; repetir la misma imagen con la misma configuración normalmente devuelve lo mismo.

Si en algún caso solo aparece el recuadro rojo, significa que el modelo no propuso una zona con el umbral elegido. Ese resultado también es útil para entender sus límites.

In [ ]:
casos_masa = [c for c in casos if c["tipo"] == "mass"]
figura, ejes = plt.subplots(2, 2, figsize=(12, 15))

for caso, eje in zip(casos_masa, ejes.flat):
    ruta = carpeta / "data/taller" / caso["imagen"]
    imagen = np.array(Image.open(ruta).convert("L"))
    entrada = np.repeat(imagen[:, :, None], 3, axis=2)
    resultado = modelo.predict(
        entrada, imgsz=1024, conf=0.05, device="cpu", verbose=False
    )[0]
    dibujar_resultado(caso, imagen, resultado, eje)
    print(f"{caso['id']}: {len(resultado.boxes)} propuesta(s)")

plt.tight_layout()
plt.show()

## Para cerrar

Ya viste la ruta completa de los datos: **PNG → tres canales → modelo → recuadros → comparación con la referencia**. En cuatro casos pueden aparecer una, varias o ninguna propuesta. Estos pocos ejemplos no permiten medir la precisión clínica del detector. El modelo no se aplicó a las calcificaciones porque fue publicado para buscar masas.

Las etiquetas internas `BIRADS45` y `BIRADS12` son categorías aprendidas por este modelo; no equivalen al resultado de patología mostrado en el cuaderno 01. Este material es didáctico y no sirve para interpretar estudios clínicos.

**Procedencia del modelo:** [Digital Eye for Mammography](https://github.com/cbddobvyz/digitaleye-mammography), [pesos públicos `shared-models.v2`](https://github.com/cbddobvyz/digitaleye-mammography/releases/tag/shared-models.v2), licencia GPL-3.0. **Procedencia de las imágenes y referencias:** [CBIS-DDSM en The Cancer Imaging Archive](https://www.cancerimagingarchive.net/collection/cbis-ddsm/), CC BY 3.0.